# 25.43 Laboratorio de Procesamiento de Señales — TP1 Estimación espectral

## GW150914

El 14 de septiembre de 2015, dos agujeros negros que llevaban mil millones de años cayéndose uno
sobre el otro [se fundieron en uno solo](https://youtu.be/Zt8Z_uzG71o?si=gQAwzKlSnAiCpfYi). La onda gravitatoria de esa fusión viajó 1300 millones de
años y atravesó la Tierra en dos décimas de segundo. La midieron dos interferómetros de cuatro
kilómetros de brazo: uno en [Hanford](https://maps.app.goo.gl/SequJrz9X3Jh53wU7) (H1) y otro en [Livingston](https://maps.app.goo.gl/pvzRZ7Ep26RGV241A) (L1), separados por 3002 km.

[Lo que midieron](https://youtu.be/TWqhUANNFXw?si=wPcfMSoKkoL0-YcT) es un estiramiento del espacio de una parte en 10²¹.

| | |
|---|---|
| fecha | 14/09/2015, 09:50:45 UTC |
| detectores | LIGO — Hanford (H1) y Livingston (L1) |
| strain | ~10⁻²¹ |
| banda | 35 a 350 Hz |
| duración | 0,2 s |

**En el registro crudo no se ve nada.** La señal está hundida en el ruido del detector, y ese
ruido es de todo menos blanco. Este TP es el trabajo que hay que hacer para que aparezca — y para
medir, al final, de qué parte del cielo vino.

---

## Qué se entrega

**Esta misma notebook, completada y corrida de arriba abajo.** No hay informe aparte: el
análisis va en las celdas de markdown, debajo de la figura que lo sostiene.

Antes de entregar, *Kernel → Restart & Run All*. Si no corre de punta a punta, no se corrige.

## Cómo se evalúa

| Criterio | Peso |
|---|---|
| **Análisis crítico** | **40 %** |
| Corrección técnica | 30 % |
| Figuras | 15 % |
| Reproducibilidad | 15 % |

**El núcleo son las cinco partes.** El núcleo bien hecho es un 8; las extensiones suman por
encima. Cinco figuras cuidadas y bien discutidas valen más que quince experimentos sin analizar.

> **Una advertencia sobre el método de trabajo.** En varias partes vas a tener que elegir
> parámetros. La manera equivocada de elegirlos es probar hasta que el gráfico quede lindo y
> escribir la justificación después. La manera correcta es decidir primero **qué necesitás que el
> estimador resuelva** y cuánta varianza tolerás, y recién entonces elegir. Se nota cuál de las
> dos hiciste.

## Las figuras

Ejes rotulados con unidades, **escala elegida** —acá casi todo es logarítmico en los dos ejes— y un
pie que diga qué hay que mirar. Una figura sin pie no argumenta: decora.

---

## Desde cuándo se puede hacer cada parte

El TP se lanza hoy y se entrega en cuatro semanas, pero no todo está habilitado desde hoy. Cada
parte dice a partir de qué clase tenés las herramientas.

| Parte | A partir de |
|---|---|
| 1. Qué hay en el registro, y cuánto registro hace falta | **Clase 2** |
| 2. Cerrá la cuenta, y estimá | **Clase 3** |
| 3. Elegí una, y defendela | **Clase 3** |
| 4. Blanquear, filtrar, ver el chirp | **Clase 3** |
| 5. Los dos detectores, y de dónde vino | **Clase 3** |
| Extensiones | cada cual elige la suya |

Es **a partir de**, no *recién en*: si llegás antes por tu cuenta, adelante.

## Los datos

Los dos detectores de LIGO alrededor de GW150914, a 4096 Hz. WOSC publica **dos ventanas** para cada evento:

| Ventana | Tiempo GPS de inicio | Archivos | Tamaño |
|---|---|---|---|
| **32 s** | 1126259447 | `H-H1_…-1126259447-32.hdf5` y su par `L-L1_…` | ~1 MB |
| **4096 s** | 1126257415 | `H-H1_…-1126257415-4096.hdf5` y su par `L-L1_…` | ~130 MB |

**Cuál te sirve es parte del trabajo, no un dato del enunciado.** Arrancá con la corta para ver de
qué se trata; la parte 1 te va a hacer calcular cuánto registro necesitás de verdad, y la parte 2
te va a hacer decidir.

El evento cae en el **tiempo GPS 1126259462,4** — a los 15,4 s de la ventana corta, y a los 2047,4 s de
la larga. Dura unos **0,2 s**. La función `cargar()` de abajo te devuelve ese instante ya
convertido al reloj del registro que pediste.

> Los detectores están a 3002 km. Una onda gravitatoria viaja a *c*, así que **el retardo entre
> los dos no puede pasar de unos 10 ms**. Guardate esa cota: es el control de sanidad de la
> parte 5.

In [2]:
# inline y no widget: la entrega tiene que renderizar sola para quien la corrija
%matplotlib inline

import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import h5py  # los registros de GWOSC son HDF5 — si falta:  pip install h5py

plt.rcParams["figure.figsize"] = (11, 3.6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Listo.")

Listo.


---

## Carga de los datos

GWOSC publica dos ventanas alrededor de cada evento: **32 s** y **4096 s**, las dos a 4096 Hz.
Cuál de las dos te sirve —y, si es la larga, cuánto de ella vas a usar— es la primera decisión del
trabajo, y la tomás en la parte 1.

La celda de abajo se encarga de todo: `cargar()` baja de GWOSC el archivo que le pidas, lo guarda en
`./datos/` y de ahí en más lo lee del disco. Cambiar de ventana es cambiar `SEGUNDOS`. La única
dependencia es `h5py` (`pip install h5py`); si estás sin conexión, copiá los archivos del Campus a
`./datos/` y la celda los toma igual.

Arrancá con los 32 s para ver de qué se trata. Después decidí.

> La ventana larga son unos 130 MB por detector, y se bajan una sola vez. Que sea un renglón de
> código no la hace gratis: lo que te cobra no es el ancho de banda, es la hipótesis de
> estacionariedad de la parte 1.

In [3]:
GPS_EVENTO = 1126259462.4
SEPARACION_KM = 3002.0
C = 299_792_458

# Las dos ventanas que GWOSC publica alrededor de GW150914, con su GPS de inicio.
VENTANAS = {32: 1126259447, 4096: 1126257415}
GWOSC = "https://gwosc.org/eventapi/json/GWTC-1-confident/GW150914/v3/"
CARPETA = "datos"


def _bajar(nombre, ruta):
    """Baja a un .parcial y recién ahí renombra: una descarga cortada no deja un
    archivo trunco haciéndose pasar por bueno."""
    os.makedirs(os.path.dirname(ruta) or ".", exist_ok=True)
    parcial = ruta + ".parcial"
    ultimo = [-1]

    def progreso(bloques, tam_bloque, total):
        if total < 10e6:  # los 32 s bajan en un parpadeo
            return
        pct = min(100, int(100 * bloques * tam_bloque / total))
        if pct >= ultimo[0] + 5:
            ultimo[0] = pct
            print(f"\r  {pct:3d} %  de {total/1e6:.0f} MB", end="", flush=True)

    print(f"Bajando {nombre} de GWOSC...")
    try:
        urllib.request.urlretrieve(GWOSC + nombre, parcial, progreso)
    except BaseException:
        if os.path.exists(parcial):
            os.remove(parcial)
        raise
    os.replace(parcial, ruta)
    if ultimo[0] >= 0:
        print()
    print(f"  guardado: {ruta}  ({os.path.getsize(ruta)/1e6:.0f} MB)")


def cargar(detector, segundos=32):
    """detector: 'H1' o 'L1'. segundos: 32 o 4096.

    Devuelve (strain, fs, t_evento), con t_evento medido desde el inicio del
    registro. La primera vez baja el archivo de GWOSC a ./datos/ y lo deja ahí;
    de ahí en más lo lee del disco.
    """
    if segundos not in VENTANAS:
        raise ValueError(f"GWOSC publica {sorted(VENTANAS)} s, no {segundos}")

    gps0 = VENTANAS[segundos]
    nombre = f"{detector[0]}-{detector}_GWOSC_4KHZ_R1-{gps0}-{segundos}.hdf5"
    ruta = os.path.join(CARPETA, nombre)

    for intento in (1, 2):
        if not os.path.exists(ruta):
            try:
                _bajar(nombre, ruta)
            except Exception as e:
                raise RuntimeError(
                    f"No se pudo bajar {nombre}. Copiá el archivo del Campus a "
                    f"./{CARPETA}/ y corré de nuevo."
                ) from e
        try:
            with h5py.File(ruta, "r") as h5:
                ds = h5["strain"]["Strain"]
                return ds[:], int(round(1 / ds.attrs["Xspacing"])), GPS_EVENTO - gps0
        except OSError:
            if intento == 2:
                raise
            print(f"  {ruta} está dañado; lo borro y lo bajo de nuevo.")
            os.remove(ruta)


# Para empezar a mirar. En la parte 1 vas a decidir si esto te alcanza.
SEGUNDOS = 32

h1, fs, t_evento = cargar("H1", SEGUNDOS)
l1, _, _ = cargar("L1", SEGUNDOS)
t = np.arange(len(h1)) / fs

print(f"{len(h1)} muestras a {fs} Hz  =  {len(h1)/fs:.0f} s por detector")
print(f"evento en t = {t_evento:.2f} s")
print(f"desvío crudo   H1: {h1.std():.3e}   L1: {l1.std():.3e}")
print()
print(f"resolución si usaras el registro entero como un solo tramo: "
      f"{fs/len(h1):.3f} Hz")
print(f"retardo máximo posible entre detectores: "
      f"{SEPARACION_KM*1e3/C*1e3:.1f} ms")

assert len(h1) == len(l1) == SEGUNDOS * fs, "largo inesperado: revisá los archivos"
assert 1e-20 < h1.std() < 1e-17, "el strain crudo debería andar en 1e-19"

Bajando H-H1_GWOSC_4KHZ_R1-1126259447-32.hdf5 de GWOSC...
  guardado: datos\H-H1_GWOSC_4KHZ_R1-1126259447-32.hdf5  (1 MB)
Bajando L-L1_GWOSC_4KHZ_R1-1126259447-32.hdf5 de GWOSC...
  guardado: datos\L-L1_GWOSC_4KHZ_R1-1126259447-32.hdf5  (1 MB)
131072 muestras a 4096 Hz  =  32 s por detector
evento en t = 15.40 s
desvío crudo   H1: 2.181e-19   L1: 2.309e-19

resolución si usaras el registro entero como un solo tramo: 0.031 Hz
retardo máximo posible entre detectores: 10.0 ms


---

# Parte 1 — Qué hay en el registro, y cuánto registro hace falta

> **A partir de la clase 2**

Los datos se bajan con la celda de arriba. Ahí empieza el trabajo:
**decir con números qué es lo que hay en ese registro**, y decidir con cuánto de él vas a trabajar.

> **Lo que estás por estimar es el ruido, no la señal.** Todo el TP descansa en un modelo de
> ruido, y el evento no es ruido. **Estimá siempre sobre tramos que no contengan el evento**, y
> decí explícitamente cuáles usaste. Es la decisión más importante de la parte 1 y la más fácil
> de pasar por alto.

## A. El primer vistazo

1. Graficá los dos registros crudos en el tiempo, marcando dónde cae el evento. ¿Se ve algo?
2. Elegí y justificá los tramos libres de evento que vas a usar de acá en adelante.
3. Calculá el periodograma de cada detector y graficalos en log–log entre 10 Hz y Nyquist.
4. **Identificá las estructuras del espectro.** Hay una forma de banda ancha y hay familias de
   líneas angostas. Para cada familia: medí su frecuencia fundamental, contá cuántos armónicos
   ves, y proponé de dónde sale. No las nombramos nosotros — encontralas.

## B. ¿Qué resolución te hace falta?

Los 32 segundos con que arrancaste **no son un dato del problema: son una elección**, y todavía
no la justificaste. Empezá por el lado que ya sabés resolver desde la clase 1.

5. Mirá la familia de líneas más apretada que encontraste en el punto 4: sus armónicos, sus
   bandas laterales si las tiene, cuán cerca están unas de otras. **¿Qué $\Delta f$ necesitás
   para separarlas del fondo y entre sí?**
6. Traducilo a muestras: con $f_s = 4096$ Hz, ¿de qué largo tiene que ser un tramo para darte ese
   $\Delta f$? Es $\Delta f = f_s / N$, de la clase 1, aplicada a una decisión real.

Anotá ese número. En la parte 2, cuando aparezca el otro lado del presupuesto, vas a poder cerrar
la cuenta y decidir cuánto registro bajar.

## C. ¿Y cuánto podés usar?

Bajar más datos parece gratis y no lo es. Todo el TP supone que el ruido es **estacionario en
sentido amplio**: que existe *una* PSD del detector y que vale para todo el tramo que uses. Un
interferómetro de cuatro kilómetros al aire libre no tiene por qué cumplir eso durante una hora.

7. Partí un tramo largo libre de evento en cuatro pedazos, estimá el periodograma de cada uno y
   comparalos. ¿Se parecen lo bastante como para que hablar de *la* PSD signifique algo?
8. Si tenés la ventana de 4096 s, hacé lo mismo con pedazos separados por varios minutos. ¿Hasta
   qué separación temporal seguís viendo el mismo ruido?

**Qué hay que responder.**

- ¿Cuántos órdenes de magnitud separan la PSD a 20 Hz de la PSD en el fondo del *bucket*?
  Ese número manda sobre todo el resto del TP.
- LIGO publica el **presupuesto de ruido** de sus detectores: qué mecanismo físico domina en cada
  banda. Buscalo y contrastalo con lo que medís. ¿Se corresponde? ¿Hay algo en tu espectro que el
  presupuesto no explique?
- El periodograma temblequea. En la clase 2 mediste cuánto y con qué distribución. ¿Cuál es el desvío
  esperado de cada punto en unidades de su propia media, y se corresponde con lo que ves?
- El evento está ahí adentro, cerca del segundo 15,4. ¿Por qué no se ve en el registro crudo?
- H1 y L1 no tienen el mismo ruido. ¿En qué se diferencian?
- **La cota de arriba y la de abajo.** Una te dice cuánto registro necesitás como mínimo; la otra,
  cuánto podés usar como máximo sin que «la PSD del detector» deje de significar algo. ¿Se cruzan?

In [4]:
# Parte 1 — tu código acá


*Tu análisis de la parte 1, acá.*

---

# Parte 2 — Cerrá la cuenta, y estimá

> **A partir de la clase 3**

La parte 1 te dejó una cuenta a medias: sabés qué $\Delta f$ necesitás, y por lo tanto de qué
largo tiene que ser cada tramo. Falta el otro lado, que es el de la clase 3.

## A. Cuántos datos, decidido

1. **Cuánta varianza tolerás.** Promediar $K$ tramos la divide por $K$. Elegí un $K$ y decí por
   qué ése: ¿qué dispersión te queda, y alcanza para lo que vas a hacer con la PSD?
2. **Multiplicá.** Largo de tramo × número de tramos = cuánto registro necesitás. Comparalo con
   los 32 s con que arrancaste. ¿Alcanzaban?
3. Si no alcanzaban, **bajá la ventana de 4096 s** y quedate con el tramo que tu cuenta pide,
   respetando la cota de estacionariedad de la parte 1.

> Ésta es la decisión que el trabajo evalúa, y las dos cotas la aprietan de verdad: la resolución
> pide tramos largos, la varianza pide muchos tramos, y la estacionariedad le pone techo al total.
> Si tu respuesta es «bajé todo», decí por qué la estacionariedad te lo permite.

## B. Los dos estimadores

Sobre el tramo de ruido de H1 que decidiste:

4. Estimá la PSD con **tu** implementación de Welch. Podés validarla contra `scipy.signal.welch`,
   pero la que se corrige es la tuya.
5. Estimá la PSD con **tu** implementación de Blackman–Tukey.
6. Graficá las dos sobre el periodograma de la parte 1, en los mismos ejes.

Para Welch los parámetros ya salieron de la cuenta de arriba: largo de tramo, número de
tramos, solapamiento y ventana. Para **B–T** la perilla es otra —el largo de retardo $M$ y la
forma de la ventana de retardo—, así que rehacé el razonamiento para ella: ¿qué $M$ te da el
$\Delta f$ que necesitás, y qué varianza te deja?

> **Dos avisos que te ahorran una noche.**
>
> La ventana de retardo de B–T no puede ser cualquiera: si su transformada no es no negativa, te
> van a salir valores de PSD **negativos** y el gráfico logarítmico va a explotar. No es un bug
> tuyo, es la clase 3.
>
> Y ojo con el rango dinámico, que ya mediste en la parte 1. Cuando abarca tantos órdenes de
> magnitud, un estimador puede tener un piso falso muy por encima del ruido verdadero. Si tu
> estimación se aplana donde el periodograma seguía bajando, no lo taparon los datos: lo tapó tu
> ventana.

In [5]:
# Parte 2 — tu código acá


*Tu análisis de la parte 2, acá. Incluí la justificación de cada parámetro.*

---

# Parte 3 — Elegí una, y defendela

> **A partir de la clase 3**

Esta parte casi no tiene código. Es la que más pesa.

Vas a blanquear con **una** de las dos estimaciones. Elegí cuál y defendé la elección con lo que
medís. Y fijate que *mejor* no significa nada por sí solo: mejor **para blanquear**, que es el
uso que le vas a dar. Ese uso es el que define el criterio.

**Qué hay que mostrar.**

1. Un zoom de las dos estimaciones alrededor de cada familia de líneas que identificaste en la
   parte 1. Ahí es donde se separan.
2. Un zoom de la zona de caída más pronunciada.
3. Una medida cuantitativa de la dispersión de cada estimación en una banda donde la PSD sea
   plana. Elegí vos la medida y decí por qué esa.

**Qué hay que responder.**

- ¿Cuál de las dos ensancha más las líneas, y por qué le pasa eso *por construcción*?
- ¿Cuál sigue mejor la pendiente de baja frecuencia?
- Para blanquear, ¿qué importa más: seguir las líneas o seguir la forma de banda ancha?
  Contestá esto **antes** de elegir, no después.
- Elegí, y decilo en una oración.

> Las dos elecciones pueden estar bien. Lo que se corrige es el argumento. Una elección defendida
> con una medición vale; la misma elección defendida con "se veía mejor", no.

In [6]:
# Parte 3 — tu código acá (poco: los zooms y la medida de dispersión)


*Tu defensa, acá. Es la sección más importante de la entrega.*

---

# Parte 4 — Blanquear, filtrar, ver el chirp

> **A partir de la clase 3**

Blanquear es dividir cada componente de frecuencia por la raíz de la PSD que estimaste: donde el
detector es ruidoso, achicás; donde es silencioso, agrandás. Después de eso todas las bandas
pesan lo mismo, y lo que sobresale sobresale de verdad.

**Qué hay que hacer**, para H1 y para L1:

1. Blanqueá el registro con la PSD que elegiste en la parte 3. **Ojo:** la PSD se estima sobre
   ruido sin evento, pero se aplica sobre el registro entero, evento incluido.
2. Recortá los bordes. El blanqueo en frecuencia ensucia las puntas; tirá un par de segundos de
   cada lado.
3. Filtrá con un pasabanda entre **35 y 350 Hz**, que es la banda donde vive la señal.
4. Graficá medio segundo alrededor de t = 15,4 s, los dos detectores.

> **Puntos de control.** Si el blanqueo salió bien, el registro blanqueado tiene **desvío ≈ 1**
> lejos del evento: es adimensional, y es la manera más rápida de saber si tu normalización está
> bien. Si te da 10⁻¹⁹ otra vez, no dividiste. Si te da 10⁵, dividiste por la PSD y no por su
> raíz. Si te da `nan`, tu PSD tiene ceros — mirá qué pasa en las puntas de la banda.
>
> El chirp dura unos **0,2 s** y barre de unos 35 Hz a unos 250 Hz, subiendo cada vez más rápido.
> Si tu ventana de tiempo es de 10 s no vas a ver nada aunque esté todo bien.

**Qué hay que responder.**

- ¿Se ve? Describí la forma de onda: qué le pasa a la frecuencia y qué le pasa a la amplitud.
- ¿Por qué el pasabanda de 35 Hz, si ya blanqueaste? ¿Qué quedaba abajo que molesta?
- Compará esta figura con la del registro crudo de la parte 1. En una oración: ¿qué hiciste?

**Y escuchalo.** `IPython.display.Audio(x, rate=fs)` sobre el tramo blanqueado y filtrado. Es
corto y grave; conviene recortar a un par de segundos alrededor del evento.

In [7]:
# Parte 4 — tu código acá


*Tu análisis de la parte 4, acá.*

---

# Parte 5 — Los dos detectores, y de dónde vino

> **A partir de la clase 3**

Hasta acá tenés el mismo evento medido dos veces, por dos instrumentos con ruidos distintos
separados por 3002 km. Eso alcanza para medir algo que ninguno de los dos podía medir solo.

**Qué hay que hacer.**

1. Estimá la **correlación cruzada** de los dos registros blanqueados y filtrados, en una ventana
   corta alrededor del evento.
2. Sacá de ahí la diferencia de tiempo de arribo Δt. Decí cuál de los dos detectores lo vio
   primero.
3. Estimá el **ángulo del cono de arribo**: la dirección de la fuente forma un ángulo θ con la
   recta que une los detectores, y ese ángulo sale de comparar *c*·Δt con los 3002 km.

> **Tres controles.**
>
> Tu Δt tiene que caer dentro de los ±10 ms que calculó la celda de carga. Si te da más, el pico
> que encontraste no es el evento.
>
> Si el pico de correlación te sale **negativo**, no es un error: los dos interferómetros están
> orientados distinto, y la misma onda los deforma con signos opuestos.
>
> Y hacé el mismo cálculo sobre un tramo **sin** evento. Si ahí también te aparece un pico
> parecido, tu Δt no significa nada.

**Qué hay que responder.**

- ¿Cuánto dio Δt, y cuál es tu incertidumbre? La resolución temporal no es infinita: ¿qué la
  limita, el muestreo o el ancho de banda de la señal?
- Con dos detectores obtenés un **cono**, no una dirección. Explicá por qué, en una oración.
- ¿Qué haría falta para angostar ese cono a una dirección?

> Esa última pregunta es el módulo 4 del curso. Guardate la respuesta que se te ocurra hoy y
> comparala en noviembre.

In [8]:
# Parte 5 — tu código acá


*Tu análisis de la parte 5, acá.*

---

# Extensiones

Suman por encima del núcleo. **Una hecha bien vale más que tres empezadas.** No hace falta
ninguna para aprobar cómodo.

**A. AR: llevá el laboratorio de C4 al TP** · *a partir de la clase 4*

En el laboratorio de la clase 4 vas a modelar el ruido de LIGO con un AR y vas a ver dónde deja de
alcanzar. Traelo acá y cerrá la pregunta que el laboratorio deja abierta: compará esa PSD con la
que elegiste en la parte 3, y decidí **si te habría servido para blanquear**. Blanqueá con ella
y mostrá el resultado al lado del de la parte 4. Si AR estima diez números en vez de quinientos
doce, ¿cuántos necesita *este* ruido, y sigue siendo buen negocio?

**B. La curva del compromiso** · *a partir de la clase 3*

Barré los parámetros de tus dos estimadores y construí la curva de resolución efectiva contra
varianza medida, con las dos series en los mismos ejes. En clase la ves sobre ruido
blanco y con seis puntos de cada estimador; acá se ve sobre tu registro, y con la
perilla barrida entera, que es la figura que hace visible el presupuesto.

**C. El promedio que usa el campo** · *a partir de la clase 3*

Los pipelines de ondas gravitatorias no promedian los periodogramas de los tramos: toman la
**mediana**. Implementalo, compará con tu Welch, y explicá qué problema resuelve la mediana que
el promedio no. Pista: mirá qué le pasa a tu estimación si un tramo con un glitch entra en la
cuenta. Conectá con la decisión que tomaste en la parte 1 sobre qué tramos usar.

**D. ¿Estacionario cuánto tiempo?** · *a partir de la clase 3*

La parte 1 comparó cuatro pedazos. Llevalo más lejos: estimá la PSD en ventanas corridas a lo
largo de los 32 s y mostrá cómo cambia. ¿Qué frecuencias son estables y cuáles se mueven?
¿Sobre qué escala de tiempo la hipótesis WSS es defendible?

---

## Antes de entregar

- *Kernel → Restart & Run All*, y que corra entero.
- Todas las figuras con ejes rotulados, unidades y pie.
- Las celdas de análisis contestadas, no vacías. **Son el 40 %.**
- Nombres de los integrantes en la celda de abajo.

**Los seis errores que más aparecen.**

1. **Usar los 32 s porque venían dados.** Son el punto de partida, no la respuesta. Si terminaste
   usándolos, que sea porque la cuenta te dio eso.
2. Estimar la PSD sobre el registro entero, evento y transitorios de borde incluidos. El evento
   es lo que buscás: no lo metas dentro de tu modelo de ruido.
3. Blanquear sin recortar los bordes, y después discutir un artefacto de borde como si fuera
   física.
4. Elegir los parámetros por lo que se ve lindo y escribir la justificación después.
5. Graficar en lineal algo que abarca diez órdenes de magnitud.
6. Reportar un Δt sin decir su incertidumbre, o sin haberlo contrastado contra un tramo sin
   evento.

---

**Grupo:**

**Integrantes:**

**Extensiones hechas:**